In [ ]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd

data = fetch_20newsgroups(
    subset="train",
    remove=("headers", "footers", "quotes"))

texts = data.data[:300]
print(texts[1:2])


["A fair number of brave souls who upgraded their SI clock oscillator have\nshared their experiences for this poll. Please send a brief message detailing\nyour experiences with the procedure. Top speed attained, CPU rated speed,\nadd on cards and adapters, heat sinks, hour of usage per day, floppy disk\nfunctionality with 800 and 1.4 m floppies are especially requested.\n\nI will be summarizing in the next two days, so please add to the network\nknowledge base if you have done the clock upgrade and haven't answered this\npoll. Thanks."]


NLP pipline

In [ ]:
from collections import Counter
import re
from nltk.corpus import stopwords
import numpy as np
import nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess(text):
  text = text.lower()
  tokens = re.findall(r'\b[a-z]+\b',text)
  return [w for w in tokens if w not in stop_words]


def build_vocab(*texts):
  vocab = set()
  for text in texts :
    tokens = preprocess(text)
    vocab.update(tokens)
  return sorted(list(vocab))


def vectorize(text, vocab):
  tokens = preprocess(text)
  counter = Counter(tokens)
  return np.array([counter.get(word,0) for word in vocab])




[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
vocab = build_vocab(*texts)
X = np.array([vectorize(text, vocab) for text in texts])

sparsity = 1 - (X > 0).mean()
density = (X > 0).mean()

print('sparsity :', sparsity)
print ('density :', density)

sparsity : 0.9924163045727291
density : 0.007583695427270846


In [ ]:
import pandas as pd

def TF(*docs):
  tf_score = []

  for doc in docs:
    tokens = preprocess(doc)
    n = len(tokens)
    counter = Counter(tokens)

    dic_doc = {}
    for word in counter :
      dic_doc[word] = counter[word]/n
    tf_score.append(dic_doc)
  return tf_score





In [ ]:
def TF_vec(*docs):
  vocab = build_vocab(*docs)
  arr_docs = []
  for doc in docs :
    tokens = preprocess(doc)
    counter = Counter(tokens)
    n = len(tokens)
    if n == 0 :
      arr_doc = np.zeros(len(vocab))
    else:
      arr_doc = np.array([counter.get(word,0)/n for word in vocab ])

    arr_docs.append(arr_doc)
  return pd.DataFrame(arr_docs, columns = vocab, index = [f'doc_{i}' for i in range(len(docs))])


TF_vec(*texts)

,aa,aardvark,ab,abandoned,abates,abbreviations,abdomens,abides,abiding,abilene,...,znkjz,zone,zoologists,zoom,zortech,zq,zubov,zum,zvb,zymmr
doc_0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
doc_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
doc_2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
doc_3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
doc_4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
doc_295,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
doc_296,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
doc_297,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
doc_298,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
def fit_tf_idf (*docs):

  N = len(docs)
  tf_matrix = TF_vec(*docs)
  vocab = list(tf_matrix.columns)

  df = (tf_matrix > 0).sum(axis = 0)
  idf_score = np.log((N + 1)/(df + 1)) + 1
  tfidf_matrix = tf_matrix.multiply(idf_score, axis=1)

  return tfidf_matrix, vocab, idf_score

tfidf_matrix, vocab, idf_score = fit_tf_idf(*texts)
print(sorted(idf_score.items(), key = lambda x:x[1], reverse = True))



[('aa', 6.01396308418893), ('aardvark', 6.01396308418893), ('ab', 6.01396308418893), ('abandoned', 6.01396308418893), ('abates', 6.01396308418893), ('abbreviations', 6.01396308418893), ('abdomens', 6.01396308418893), ('abides', 6.01396308418893), ('abiding', 6.01396308418893), ('abilene', 6.01396308418893), ('ability', 6.01396308418893), ('ably', 6.01396308418893), ('aboard', 6.01396308418893), ('abode', 6.01396308418893), ('abolish', 6.01396308418893), ('abort', 6.01396308418893), ('abortion', 6.01396308418893), ('abp', 6.01396308418893), ('abs', 6.01396308418893), ('absence', 6.01396308418893), ('abstract', 6.01396308418893), ('absurd', 6.01396308418893), ('abuses', 6.01396308418893), ('academy', 6.01396308418893), ('accel', 6.01396308418893), ('accelerated', 6.01396308418893), ('acceleration', 6.01396308418893), ('acceptance', 6.01396308418893), ('accessible', 6.01396308418893), ('accessories', 6.01396308418893), ('accessory', 6.01396308418893), ('accidental', 6.01396308418893), ('a

In [ ]:
def vectorize_query(query, vocab, idf_score):
  tokens = preprocess(query)
  counter = Counter(tokens)
  n = len(tokens)
  if n == 0 :
    tf_query = np.zeros(len(vocab))
  else:
    tf_query = np.array([counter.get(word,0)/n for word in vocab ])
  return tf_query * idf_score.to_numpy()

query = "machine learning artificial intelligence"
t1 = 'ai and machine learning are the new tech'
t2 = 'as a data scientist, i do explore tech news'
t3 = 'what are the algorithms used for classification'

tfidf_matrix1, vocab1, idf_score1 = fit_tf_idf(query, t1, t2, t3)
vectorize_query(query,vocab1,idf_score1)

array([0.        , 0.        , 0.47907268, 0.        , 0.        ,
       0.        , 0.47907268, 0.37770641, 0.37770641, 0.        ,
       0.        , 0.        , 0.        , 0.        ])

In [ ]:
tokens = preprocess(query)
print(tokens)
counter = Counter(tokens)
print(counter)
n = len(tokens)
if n == 0 :
  tf_query = np.zeros(len(vocab))
else:
  tf_query = np.array([counter.get(word,0)/n for word in vocab ])
print(f'tf query = {tf_query}\nidf score = {idf_score}')
print('tf idf = ',tf_query * idf_score.to_numpy())

['machine', 'learning', 'artificial', 'intelligence']
Counter({'machine': 1, 'learning': 1, 'artificial': 1, 'intelligence': 1})
tf query = [0. 0. 0. ... 0. 0. 0.]
idf score = aa           6.013963
aardvark     6.013963
ab           6.013963
abandoned    6.013963
abates       6.013963
               ...   
zq           6.013963
zubov        6.013963
zum          6.013963
zvb          6.013963
zymmr        6.013963
Length: 8055, dtype: float64
tf idf =  [0. 0. 0. ... 0. 0. 0.]


In [ ]:
def cosine_similarity(v1, v2) :
  dot = np.dot(v1, v2)
  n1 = np.linalg.norm(v1)
  n2 = np.linalg.norm(v2)
  if n1 == 0 or n2 == 0 :
    return 0
  return dot/(n1*n2)

In [ ]:
tfidf_matrix, vocab, idf_score = fit_tf_idf(*texts)

def search(query, docs, tfidf_matrix, vocab, idf_score, top_k=3):
    query_vec = vectorize_query(query, vocab, idf_score)

    scores = []

    for i in range(len(docs)):
        doc_vec = tfidf_matrix.iloc[i].to_numpy()
        score = cosine_similarity(query_vec, doc_vec)
        scores.append((i, score, docs[i]))

    return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

In [ ]:
query = "machine learning artificial intelligence"

results = search(query, texts,tfidf_matrix, vocab, idf_score, 3)

for _, score, text in results:
    print(score)
    print(text[:200])
    print("-----")

0.0993423016922807
I have been playing with my Centris 610 for almost a week now.  I must say
this machine is really fast!  The hardware turn-on feature is annoying, but
I got PowerKey from Sophisicated Circuits and it 
-----
0.07558471496284289
Is it possible to use WinQVT/Net on a machine that uses NDIS to connect to a
Token Ring ? I tried it with older versions (< 3.2) but got an invalid packet
class error or something the like...

Regards
-----
0.0725952841143644
You have a lot more problems keeping up with hardware interrupts in Windows than
in DOS - regardless of what communication software you are using.

Try the following:
   1) Turn off disk write cache f
-----


In [ ]:
def normalize_matrix(matrix):
    L2 = np.linalg.norm(matrix, ord=2, axis=1)
    return matrix.div(L2.replace(0, 1), axis=0)

In [ ]:
'''def fit_tf_idf(*docs):
    N = len(docs)

    tf_matrix = TF_vec(*docs)
    vocab = list(tf_matrix.columns)

    df = (tf_matrix > 0).sum(axis=0)
    idf_score = np.log((N + 1) / (df + 1)) + 1

    tfidf_matrix = tf_matrix.multiply(idf_score, axis=1)

    tfidf_matrix = normalize_matrix(tfidf_matrix)

    return tfidf_matrix, vocab, idf_score'''

'def fit_tf_idf(*docs):\n    N = len(docs)\n\n    tf_matrix = TF_vec(*docs)\n    vocab = list(tf_matrix.columns)\n\n    df = (tf_matrix > 0).sum(axis=0)\n    idf_score = np.log((N + 1) / (df + 1)) + 1\n\n    tfidf_matrix = tf_matrix.multiply(idf_score, axis=1)\n\n    tfidf_matrix = normalize_matrix(tfidf_matrix)\n\n    return tfidf_matrix, vocab, idf_score'

In [ ]:
def vectorize_query(query, vocab, idf_score):
    tokens = preprocess(query)
    counter = Counter(tokens)
    n = len(tokens)

    if n == 0:
        query_vec = np.zeros(len(vocab))
    else:
        query_vec = np.array([counter.get(word, 0) / n for word in vocab])

    query_vec = query_vec * idf_score.to_numpy()

    norm = np.linalg.norm(query_vec)

    if norm == 0:
        return query_vec

    return query_vec / norm

In [ ]:
def search(query, docs, tfidf_matrix, vocab, idf_score, top_k=3):
    query_vec = vectorize_query(query, vocab, idf_score)

    scores = tfidf_matrix.to_numpy() @ query_vec

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for i in ranked_indices:
        results.append((i, scores[i], docs[i]))

    return results


In [ ]:
def fit_tf_idf(*docs, min_df=2, max_df=0.8):
    N = len(docs)

    tf_matrix = TF_vec(*docs)

    df = (tf_matrix > 0).sum(axis=0)

    if isinstance(max_df, float):
        max_df_value = max_df * N
    else:
        max_df_value = max_df

    selected_words = df[(df >= min_df) & (df <= max_df_value)].index

    tf_matrix = tf_matrix[selected_words]

    vocab = list(tf_matrix.columns)

    df = (tf_matrix > 0).sum(axis=0)

    idf_score = np.log((N + 1) / (df + 1)) + 1

    tfidf_matrix = tf_matrix.multiply(idf_score, axis=1)

    tfidf_matrix = normalize_matrix(tfidf_matrix)<<

    return tfidf_matrix, vocab, idf_score

In [ ]:
tfidf_matrix, vocab, idf_score = fit_tf_idf(*texts, min_df=2, max_df=0.8)

AttributeError: 'numpy.ndarray' object has no attribute 'replace'

In [ ]:
def fit_tf_idf(*docs, min_df=2, max_df=0.8, top_k_features=None):
    N = len(docs)

    tf_matrix = TF_vec(*docs)

    df = (tf_matrix > 0).sum(axis=0)

    if isinstance(max_df, float):
        max_df_value = max_df * N
    else:
        max_df_value = max_df

    selected_words = df[(df >= min_df) & (df <= max_df_value)].index
    tf_matrix = tf_matrix[selected_words]

    df = (tf_matrix > 0).sum(axis=0)
    idf_score = np.log((N + 1) / (df + 1)) + 1

    if top_k_features is not None:
        selected_words = idf_score.sort_values(ascending=False).head(top_k_features).index
        tf_matrix = tf_matrix[selected_words]
        idf_score = idf_score[selected_words]

    tfidf_matrix = tf_matrix.multiply(idf_score, axis=1)
    tfidf_matrix = normalize_matrix(tfidf_matrix)

    vocab = list(tfidf_matrix.columns)

    return tfidf_matrix, vocab, idf_score

In [ ]:
tfidf_matrix, vocab, idf_score = fit_tf_idf(
    *texts,
    min_df=2,
    max_df=0.8,
    top_k_features=1000)